In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import transforms, datasets
import torch.nn as nn
from google.colab import drive
from torch.utils.data import DataLoader
import torch.nn.functional as F
from tqdm import tqdm

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
from torch.cuda.random import manual_seed

# Load data from "Data12"

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    # --- New Augmentations ---
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    # -------------------------
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


train_data = datasets.ImageFolder(root='/content/drive/MyDrive/Assignment_2_Dataset/Task 1 & 2/Data 12/train', transform=transform)

test_data = datasets.ImageFolder(root='/content/drive/MyDrive/Assignment_2_Dataset/Task 1 & 2/Data 12/test', transform=test_transform)

torch.manual_seed(42)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

torch.manual_seed(42)

test_loader = DataLoader(test_data, batch_size=32, shuffle=True)

In [ ]:
# def get_mean_and_std_gpu(loader):
#     mean = torch.zeros(3).to(device) # Initialize on GPU
#     std = torch.zeros(3).to(device)
#     total_images_count = 0

#     for images, _ in loader:
#         # 2. Move images to GPU
#         images = images.to(device)

#         # Flatten: [Batch, Channels, Height * Width]
#         images = images.view(images.size(0), images.size(1), -1)

#         # 3. Perform math on the GPU
#         mean += images.mean(2).sum(0)
#         std += images.std(2).sum(0)
#         total_images_count += images.size(0)

#     mean /= total_images_count
#     std /= total_images_count

#     # Move back to CPU to print nicely if you want
#     return mean.cpu(), std.cpu()

# mean, std = get_mean_and_std_gpu(loader)
# print(f"Mean: {mean}, Std: {std}")

In [ ]:
class CNN(nn.Module):
    def __init__(self,num_classes):
        super().__init__()
        self.model = nn.Sequential(
            # CL1
            nn.Conv2d(3, 4, kernel_size=3, stride=1, padding=1),

            nn.ReLU(),
            # PL1
            nn.AvgPool2d(kernel_size=2, stride=2),


            # CL2
            nn.Conv2d(4, 8, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),

            # PL2
            nn.AvgPool2d(kernel_size=2, stride=2)
        )

        # --- MLFNN Classifier Block ---
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(8 * 16 * 16, 512),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(p=0.1),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.model(x)
        x = self.classifier(x)
        return x



In [ ]:
# Initialize the model we defined earlier
model = CNN(num_classes=5).to(device)
# Loss Function and Optimizer
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
len(train_data.samples)

3500

In [ ]:
num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    # Use enumerate to get the iteration count 'i' automatically
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step() # Fixed the typo here

        running_loss += loss.item()

        # Progress Calculation: (Current Batch / Total Batches) * 100
        progress = (i + 1) / len(train_loader) * 100
        if (i + 1) % 5 == 0: # Print every 5 batches so it doesn't spam the screen
            print(f"Epoch {epoch+1} | Progress: {progress:.2f}% | Loss: {loss.item():.4f}")

    print(f"==> Epoch [{epoch+1}/{num_epochs}] Finished. Avg Loss: {running_loss/len(train_loader):.4f}")

Epoch 1 | Progress: 4.55% | Loss: 1.6201
Epoch 1 | Progress: 9.09% | Loss: 1.5915
Epoch 1 | Progress: 13.64% | Loss: 1.6044
Epoch 1 | Progress: 18.18% | Loss: 1.6259
Epoch 1 | Progress: 22.73% | Loss: 1.6117
Epoch 1 | Progress: 27.27% | Loss: 1.5973
Epoch 1 | Progress: 31.82% | Loss: 1.5971
Epoch 1 | Progress: 36.36% | Loss: 1.6104
Epoch 1 | Progress: 40.91% | Loss: 1.6655
Epoch 1 | Progress: 45.45% | Loss: 1.5989
Epoch 1 | Progress: 50.00% | Loss: 1.5708
Epoch 1 | Progress: 54.55% | Loss: 1.5684
Epoch 1 | Progress: 59.09% | Loss: 1.5192
Epoch 1 | Progress: 63.64% | Loss: 1.5411
Epoch 1 | Progress: 68.18% | Loss: 1.5436
Epoch 1 | Progress: 72.73% | Loss: 1.4761
Epoch 1 | Progress: 77.27% | Loss: 1.5162
Epoch 1 | Progress: 81.82% | Loss: 1.4585
Epoch 1 | Progress: 86.36% | Loss: 1.2848
Epoch 1 | Progress: 90.91% | Loss: 1.4677
Epoch 1 | Progress: 95.45% | Loss: 1.4823
Epoch 1 | Progress: 100.00% | Loss: 1.5695
==> Epoch [1/50] Finished. Avg Loss: 1.5591
Epoch 2 | Progress: 4.55% | Loss:

In [ ]:
model.eval() # Set model to evaluation mode
correct = 0
total = 0

with torch.no_grad(): # No need to track gradients during testing
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)

        # The class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        print("-")

accuracy = 100 * correct / total
print(f'\nFinal Accuracy on the test images: {accuracy:.2f}%')

-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-

Final Accuracy on the test images: 70.86%


In [ ]:
model.eval() # Set model to evaluation mode
correct = 0
total = 0

with torch.no_grad(): # No need to track gradients during testing
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)

        # The class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'\nFinal Accuracy on the train images: {accuracy:.2f}%')


Final Accuracy on the train images: 79.46%
